In [ ]:
import pandas as pd
import numpy as np

import scipy.stats as stats

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Modern FunctionTransformer usage
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

# Enable pandas output for transformers (sklearn >= 1.2)
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
df = pd.read_csv('train.csv', usecols=['Age', 'Fare', 'Survived'])

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
# Fill missing Age with median (more robust to outliers than mean)
df['Age'].fillna(df['Age'].median(), inplace=True)

In [ ]:
df.head()

In [ ]:
X = df[['Age', 'Fare']]
y = df['Survived']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Visualise Age distribution before transformation
plt.figure(figsize=(14, 4))
plt.subplot(121)
sns.histplot(X_train['Age'], kde=True)  # histplot replaces deprecated distplot
plt.title('Age PDF')

plt.subplot(122)
stats.probplot(X_train['Age'], dist="norm", plot=plt)
plt.title('Age QQ Plot')

plt.show()

In [ ]:
# Visualise Fare distribution before transformation
plt.figure(figsize=(14, 4))
plt.subplot(121)
sns.histplot(X_train['Fare'], kde=True)  # histplot replaces deprecated distplot
plt.title('Fare PDF')

plt.subplot(122)
stats.probplot(X_train['Fare'], dist="norm", plot=plt)
plt.title('Fare QQ Plot')

plt.show()

In [ ]:
clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

In [ ]:
clf.fit(X_train, y_train)
clf2.fit(X_train, y_train)
    
y_pred = clf.predict(X_test)
y_pred1 = clf2.predict(X_test)
    
print("Accuracy LR", accuracy_score(y_test, y_pred))
print("Accuracy DT", accuracy_score(y_test, y_pred1))

In [ ]:
# FunctionTransformer with modern API
# feature_names_out='one-to-one' preserves column names when transform_output='pandas'
trf = FunctionTransformer(func=np.log1p, feature_names_out='one-to-one')

In [ ]:
X_train_transformed = trf.fit_transform(X_train)
X_test_transformed = trf.transform(X_test)

In [ ]:
clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

clf.fit(X_train_transformed, y_train)
clf2.fit(X_train_transformed, y_train)
    
y_pred = clf.predict(X_test_transformed)
y_pred1 = clf2.predict(X_test_transformed)
    
print("Accuracy LR", accuracy_score(y_test, y_pred))
print("Accuracy DT", accuracy_score(y_test, y_pred1))

In [ ]:
# Cross-validation on log-transformed features
X_transformed = trf.fit_transform(X)

clf = LogisticRegression()
clf2 = DecisionTreeClassifier()

print("LR", np.mean(cross_val_score(clf, X_transformed, y, scoring='accuracy', cv=10)))
print("DT", np.mean(cross_val_score(clf2, X_transformed, y, scoring='accuracy', cv=10)))

In [ ]:
# QQ plots: Fare before and after log transformation
plt.figure(figsize=(14, 4))

plt.subplot(121)
stats.probplot(X_train['Fare'], dist="norm", plot=plt)
plt.title('Fare Before Log')

plt.subplot(122)
stats.probplot(X_train_transformed['Fare'], dist="norm", plot=plt)
plt.title('Fare After Log')

plt.show()

In [ ]:
# QQ plots: Age before and after log transformation
plt.figure(figsize=(14, 4))

plt.subplot(121)
stats.probplot(X_train['Age'], dist="norm", plot=plt)
plt.title('Age Before Log')

plt.subplot(122)
stats.probplot(X_train_transformed['Age'], dist="norm", plot=plt)
plt.title('Age After Log')

plt.show()

In [ ]:
# Using FunctionTransformer inside a ColumnTransformer
# Apply log1p only to Fare column, pass Age through unchanged
trf_col = ColumnTransformer([
    ('log_fare', FunctionTransformer(func=np.log1p, feature_names_out='one-to-one'), ['Fare'])
], remainder='passthrough', verbose_feature_names_out=False)

In [ ]:
X_train_col = trf_col.fit_transform(X_train)
X_test_col = trf_col.transform(X_test)

print(X_train_col.head())

In [ ]:
clf = LogisticRegression()
clf.fit(X_train_col, y_train)

print("Accuracy LR (Fare log, Age unchanged)", accuracy_score(y_test, clf.predict(X_test_col)))